In [1]:
from pyesgf.search import SearchConnection

required_vars = ["tos", "sos", "hfds", "wfo", "areacello", 'msftmz']

conn = SearchConnection("https://esgf-node.llnl.gov/esg-search", distrib=True)

ctx = conn.new_context(
    project="CMIP6",
    experiment_id="piControl",
    variable_id=",".join(required_vars),   # multiple variables in one query
    facets="source_id",
    latest=True
)

models = sorted(ctx.facet_counts["source_id"].keys())

print("Models with required variables in piControl:")
print(models)

Models with required variables in piControl:
['ACCESS-CM2', 'ACCESS-ESM1-5', 'AWI-CM-1-1-MR', 'AWI-ESM-1-1-LR', 'BCC-CSM2-MR', 'BCC-ESM1', 'CAMS-CSM1-0', 'CAS-ESM2-0', 'CESM2', 'CESM2-FV2', 'CESM2-WACCM', 'CESM2-WACCM-FV2', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'CanESM5', 'CanESM5-1', 'CanESM5-CanOE', 'E3SM-1-0', 'E3SM-1-1', 'E3SM-1-1-ECA', 'E3SM-2-0', 'E3SM-2-0-NARRM', 'E3SM-2-1', 'EC-Earth3', 'EC-Earth3-AerChem', 'EC-Earth3-CC', 'EC-Earth3-ESM-1', 'EC-Earth3-LR', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'FGOALS-f3-L', 'FGOALS-g3', 'FIO-ESM-2-0', 'GFDL-CM4', 'GFDL-ESM4', 'GISS-E2-1-G', 'GISS-E2-1-G-CC', 'GISS-E2-1-H', 'GISS-E2-2-G', 'GISS-E2-2-H', 'GISS-E3-G', 'HadGEM3-GC31-LL', 'HadGEM3-GC31-MM', 'ICON-ESM-LR', 'IITM-ESM', 'INM-CM4-8', 'INM-CM5-0', 'IPSL-CM5A2-INCA', 'IPSL-CM6A-LR', 'IPSL-CM6A-MR1', 'KIOST-ESM', 'MCM-UA-1-0', 'MIROC-ES2H', 'MIROC-ES2L', 'MIROC6', 'MPI-ESM-1-2-HAM', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR', 'MRI-ESM2-0', 'NESM3', 'Nor

In [19]:
import intake
import xarray as xr
import matplotlib.pyplot as plt
import warnings

# Mute the annoying cftime serialization warnings
warnings.simplefilter("ignore", category=xr.SerializationWarning)

# 1. Connect to the catalog
url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
col = intake.open_esm_datastore(url)

# 2. Search for the exact CanESM5-1 dataset
query = col.search(
    source_id="BCC-ESM1",
    experiment_id="piControl",
    member_id="r1i1p1f1",
    variable_id="tos",
    table_id="Omon"
)

print(f"Found {len(query.df)} matching assets in the catalog.")

# 3. Load it lazily into xarray
# We don't strictly need the 'override' arguments here since we only 
# have one variable, but it's good practice to keep them for CMIP6!
dset_dict = query.to_dataset_dict(
    zarr_kwargs={'consolidated': True},
    xarray_combine_by_coords_kwargs={'compat': 'override', 'join': 'override'}
)

# 4. Extract the actual dataset from the dictionary
dataset_key = list(dset_dict.keys())[0]
ds = dset_dict[dataset_key]

print(f"\nSuccessfully loaded: {dataset_key}")

Found 1 matching assets in the catalog.

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'



Successfully loaded: CMIP.BCC.BCC-ESM1.piControl.Omon.gn


In [20]:
list(dset_dict.keys())

['CMIP.BCC.BCC-ESM1.piControl.Omon.gn']

In [21]:
dset_dict['CMIP.BCC.BCC-ESM1.piControl.Omon.gn']

<xarray.Dataset> Size: 2GB
Dimensions:         (lat: 232, bnds: 2, lon: 360, time: 5412, member_id: 1,
                     dcpp_init_year: 1)
Coordinates:
  * lat             (lat) float64 2kB -81.5 -80.5 -79.5 -78.5 ... 87.5 88.5 89.5
    lat_bnds        (lat, bnds) float64 4kB dask.array<chunksize=(232, 2), meta=np.ndarray>
    latitude        (lat, lon) float32 334kB dask.array<chunksize=(232, 360), meta=np.ndarray>
  * lon             (lon) float64 3kB 0.5 1.5 2.5 3.5 ... 357.5 358.5 359.5
    lon_bnds        (lon, bnds) float64 6kB dask.array<chunksize=(360, 2), meta=np.ndarray>
    longitude       (lat, lon) float32 334kB dask.array<chunksize=(232, 360), meta=np.ndarray>
  * time            (time) object 43kB 1850-01-16 12:00:00 ... 2300-12-16 12:...
    time_bnds       (time, bnds) object 87kB dask.array<chunksize=(5412, 2), meta=np.ndarray>
  * member_id       (member_id) object 8B 'r1i1p1f1'
  * dcpp_init_year  (dcpp_init_year) float64 8B nan
Dimensions without coordinates: bnds
Data variables:
    tos             (member_id, dcpp_init_year, time, lat, lon) float32 2GB dask.array<chunksize=(1, 1, 299, 232, 360), meta=np.ndarray>
Attributes: (12/65)
    Conventions:                      CF-1.7 CMIP-6.2
    activity_id:                      CMIP
    branch_method:                    standard
    branch_time_in_child:             0.0
    branch_time_in_parent:            0.0
    cmor_version:                     3.3.2
    ...                               ...
    intake_esm_attrs:variable_id:     tos
    intake_esm_attrs:grid_label:      gn
    intake_esm_attrs:zstore:          gs://cmip6/CMIP6/CMIP/BCC/BCC-ESM1/piCo...
    intake_esm_attrs:version:         20181218
    intake_esm_attrs:_data_format_:   zarr
    intake_esm_dataset_key:           CMIP.BCC.BCC-ESM1.piControl.Omon.gn